# TP 5 — Solutions: Files and Exception Handling

**Module**: Programmation Python — ELNI 5.5  
**For instructor use / post-lab release**


In [ ]:
import os

NOTEBOOK_DIR = os.getcwd()
ASSETS_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'assets')
OUTPUT_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'output')
CSV_PATH = os.path.join(ASSETS_DIR, 'sensor_data.csv')
LOG_PATH = os.path.join(ASSETS_DIR, 'sample_log.txt')
os.makedirs(OUTPUT_DIR, exist_ok=True)

---

## Solution — Exercise 1: Directory Navigation


In [ ]:
import os

# 1. CWD
print(f"CWD: {os.getcwd()}")

# 2. List assets/
assets_contents = os.listdir(ASSETS_DIR)
print(f"\nassets/ contains: {assets_contents}")

# 3. Type and size for each item
print("\nDetailed listing:")
for item in sorted(assets_contents):
    full_path = os.path.join(ASSETS_DIR, item)
    if os.path.isfile(full_path):
        size = os.path.getsize(full_path)
        print(f"  FILE : {item:<30} ({size} bytes)")
    elif os.path.isdir(full_path):
        print(f"  DIR  : {item}")

# 4. Check for missing file
missing = os.path.join(ASSETS_DIR, 'missing_data.csv')
if os.path.exists(missing):
    print(f"\nmissing_data.csv found.")
else:
    print(f"\nmissing_data.csv does NOT exist in assets/ — as expected.")

# 5. Build path and get absolute path
csv_path = os.path.join(ASSETS_DIR, 'sensor_data.csv')
abs_path = os.path.abspath(csv_path)
print(f"\nAbsolute CSV path: {abs_path}")

---

## Solution — Exercise 2: Reading the Log File


In [ ]:
# 1. Read all lines
with open(LOG_PATH, 'r', encoding='utf-8') as f:
    log_lines = f.readlines()

# 2. Line count
print(f"Total lines: {len(log_lines)}")

# 3. ERROR and WARNING lines
print("\nError/Warning lines:")
for line in log_lines:
    if 'ERROR' in line or 'WARNING' in line:
        print(f"  {line.strip()}")

# 4. Count by level
info_count = sum(1 for l in log_lines if 'INFO' in l)
warn_count = sum(1 for l in log_lines if 'WARNING' in l)
err_count = sum(1 for l in log_lines if 'ERROR' in l)

print(f"\nLog level summary:")
print(f"  INFO    : {info_count}")
print(f"  WARNING : {warn_count}")
print(f"  ERROR   : {err_count}")

# 5. Extract timestamp from first line
first_line = log_lines[0].strip()
# Format: [2024-01-15 08:00:00] INFO  ...
timestamp = first_line[1:20]  # skip '[', take 19 chars
print(f"\nFirst timestamp: {timestamp}")

---

## Solution — Exercise 3: Reading and Parsing CSV


In [ ]:
import csv

sensor_records = []
fieldnames = None

with open(CSV_PATH, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    for row in reader:
        sensor_records.append({
            "timestamp": row["timestamp"],
            "sensor_id": row["sensor_id"],
            "temperature_c": float(row["temperature_C"]),
            "pressure_kpa": float(row["pressure_kPa"]),
            "status": row["status"]
        })

# 3. Header fields
print(f"CSV columns: {fieldnames}")

# 4. Formatted table
print(f"\n{'Timestamp':<22} {'Sensor':<8} {'Temp (°C)':>10} {'Pressure (kPa)':>16} {'Status':<8}")
print("-" * 68)
for r in sensor_records:
    temp_s = f"{r['temperature_c']:>10.1f}" if r['status'] == 'OK' else f"{'ERR':>10}"
    print(f"{r['timestamp']:<22} {r['sensor_id']:<8} {temp_s} {r['pressure_kpa']:>16.2f} {r['status']:<8}")

# 5. Counts
total = len(sensor_records)
errors = sum(1 for r in sensor_records if r['status'] == 'ERROR')
ok = total - errors
print(f"\nTotal: {total} | OK: {ok} | ERROR: {errors}")

---

## Solution — Exercise 4: Writing Variables to a File


In [ ]:
import os

# 1. Compute statistics
stats = {}
for r in sensor_records:
    sid = r["sensor_id"]
    if r["status"] != "OK" or r["temperature_c"] < -100:
        continue
    if sid not in stats:
        stats[sid] = []
    stats[sid].append(r["temperature_c"])

# 2. Write report
report_path = os.path.join(OUTPUT_DIR, 'sensor_report.txt')

with open(report_path, 'w', encoding='utf-8') as f:
    f.write("Sensor Monitoring Report\n")
    f.write("=" * 40 + "\n")
    f.write(f"Total records   : {len(sensor_records)}\n")
    f.write(f"Error records   : {sum(1 for r in sensor_records if r['status']=='ERROR')}\n")
    f.write("\n")

    for sid, temps in sorted(stats.items()):
        n = len(temps)
        avg = sum(temps) / n
        f.write(f"Sensor {sid}:\n")
        f.write(f"  Valid readings  : {n}\n")
        f.write(f"  Average temp    : {avg:.2f} °C\n")
        f.write(f"  Min temperature : {min(temps):.1f} °C\n")
        f.write(f"  Max temperature : {max(temps):.1f} °C\n")
        f.write("\n")

# 3. Read back
with open(report_path, 'r', encoding='utf-8') as f:
    print(f.read())

# 4. Append footer
with open(report_path, 'a', encoding='utf-8') as f:
    f.write("Report generated: 2024-01-15\n")

print("Footer appended.")

---

## Solution — Exercise 5: Copying Files


In [ ]:
import shutil
import os

# 1. Copy sensor_data.csv
csv_backup = os.path.join(OUTPUT_DIR, 'sensor_data.csv')
shutil.copy(CSV_PATH, csv_backup)
print(f"Copied to: {csv_backup}")

# 2. Verify
print(f"Copy exists: {os.path.exists(csv_backup)}")

# 3. Compare sizes
orig_size = os.path.getsize(CSV_PATH)
copy_size = os.path.getsize(csv_backup)
print(f"Original : {orig_size} bytes")
print(f"Copy     : {copy_size} bytes")
print(f"Sizes match: {orig_size == copy_size}")

# 4. Copy log with shutil.copy2
log_backup = os.path.join(OUTPUT_DIR, 'sample_log_backup.txt')
shutil.copy2(LOG_PATH, log_backup)
print(f"\nLog backed up to: {log_backup}")

# 5. List output/
print(f"\nOutput directory contents:")
for item in sorted(os.listdir(OUTPUT_DIR)):
    full = os.path.join(OUTPUT_DIR, item)
    size = os.path.getsize(full) if os.path.isfile(full) else 0
    print(f"  {item:<35} {size:>8} bytes")

---

## Solution — Exercise 6: Robust File Reader


In [ ]:
import csv
import os


def safe_parse_float(value_str, field_name):
    """
    Parse a string to float, returning None on failure.
    """
    try:
        return float(value_str)
    except ValueError:
        print(f"    Warning: cannot convert {value_str!r} to float for field '{field_name}'")
        return None


def safe_read_csv(filepath):
    """
    Safely read sensor CSV data with full exception handling.
    """
    records = []
    filename = os.path.basename(filepath)

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for line_num, row in enumerate(reader, start=2):
                try:
                    temp = safe_parse_float(row.get("temperature_C", ""), "temperature_C")
                    pres = safe_parse_float(row.get("pressure_kPa", ""), "pressure_kPa")
                    records.append({
                        "timestamp": row.get("timestamp", ""),
                        "sensor_id": row.get("sensor_id", ""),
                        "temperature_c": temp,
                        "pressure_kpa": pres,
                        "status": row.get("status", "")
                    })
                except Exception as e:
                    print(f"  Skipping line {line_num}: {e}")

    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return []
    except PermissionError:
        print(f"Permission denied: {filepath}")
        return []
    else:
        print(f"Successfully read {len(records)} records from {filename}.")
    finally:
        print(f"Done attempting to read {filename}.")

    return records


print("=== Test 1: valid file ===")
result = safe_read_csv(CSV_PATH)
print(f"Records returned: {len(result)}")

print("\n=== Test 2: missing file ===")
result2 = safe_read_csv("nonexistent_file.csv")
print(f"Records returned: {len(result2)}")

---

## Solution — Exercise 7: Raising Exceptions


In [ ]:
import csv
import os


def write_csv_report(records, output_filepath):
    """
    Write a list of sensor records to a CSV file with input validation.
    """
    if not isinstance(records, list):
        raise TypeError(f"records must be a list, got {type(records).__name__}")
    if len(records) == 0:
        raise ValueError("records list is empty — nothing to write")
    if not output_filepath.endswith('.csv'):
        raise ValueError(f"output_filepath must end with '.csv', got {output_filepath!r}")

    # Determine fieldnames from the first record
    fieldnames = list(records[0].keys())

    os.makedirs(os.path.dirname(output_filepath) or '.', exist_ok=True)

    with open(output_filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(records)

    return len(records)


test_output = os.path.join(OUTPUT_DIR, 'valid_output.csv')

print("Test 1 — Valid records:")
try:
    n = write_csv_report(sensor_records, test_output)
    print(f"  Written {n} records to {test_output}")
except (ValueError, TypeError) as e:
    print(f"  Error: {e}")

print("\nTest 2 — Empty list:")
try:
    write_csv_report([], test_output)
except (ValueError, TypeError) as e:
    print(f"  Error: {e}")

print("\nTest 3 — Wrong extension:")
try:
    write_csv_report(sensor_records, 'output.txt')
except (ValueError, TypeError) as e:
    print(f"  Error: {e}")

print("\nTest 4 — Non-list records:")
try:
    write_csv_report(42, test_output)
except (ValueError, TypeError) as e:
    print(f"  Error: {e}")

---

## Solution — Stretch Goal: End-to-End Pipeline


In [ ]:
import csv
import os
import shutil


def run_monitoring_pipeline(csv_path, log_path, output_dir):
    """
    Full end-to-end monitoring data pipeline.

    Reads CSV and log files, processes data, writes report, backs up files.
    """
    os.makedirs(output_dir, exist_ok=True)
    report_path = os.path.join(output_dir, 'pipeline_report.txt')
    summary = {"status": "unknown", "records_loaded": 0, "valid_records": 0}

    try:
        # Step 1: Load CSV
        records = safe_read_csv(csv_path)
        summary["records_loaded"] = len(records)

        # Step 2: Read log file
        log_summary = {"INFO": 0, "WARNING": 0, "ERROR": 0}
        with open(log_path, 'r', encoding='utf-8') as f:
            for line in f:
                for level in log_summary:
                    if level in line:
                        log_summary[level] += 1

        # Step 3: Filter valid records
        valid = [r for r in records if r["status"] == "OK" and
                 r["temperature_c"] is not None and r["temperature_c"] > -100]
        summary["valid_records"] = len(valid)

        # Step 4: Statistics per sensor
        sensor_stats = {}
        for r in valid:
            sid = r["sensor_id"]
            sensor_stats.setdefault(sid, []).append(r["temperature_c"])

        # Step 5: Detect large temperature jumps (>1°C between consecutive readings)
        jumps = []
        for sid, temps in sensor_stats.items():
            for i in range(1, len(temps)):
                delta = abs(temps[i] - temps[i - 1])
                if delta > 1.0:
                    jumps.append({"sensor": sid, "index": i, "delta": delta})

        # Step 6: Write report
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("Monitoring Pipeline Report\n")
            f.write("=" * 40 + "\n")
            f.write(f"Total records   : {len(records)}\n")
            f.write(f"Valid records   : {len(valid)}\n")
            f.write(f"Error records   : {len(records)-len(valid)}\n\n")

            f.write("Log Summary:\n")
            for level, count in log_summary.items():
                f.write(f"  {level:<10}: {count}\n")
            f.write("\n")

            f.write("Temperature Statistics:\n")
            for sid, temps in sorted(sensor_stats.items()):
                n = len(temps)
                avg = sum(temps) / n
                f.write(f"  Sensor {sid}: n={n}, avg={avg:.2f}, "
                        f"min={min(temps):.1f}, max={max(temps):.1f} °C\n")
            f.write("\n")

            if jumps:
                f.write("Temperature Jumps (>1 °C):\n")
                for j in jumps:
                    f.write(f"  Sensor {j['sensor']}: Δ={j['delta']:.2f} °C at reading #{j['index']+1}\n")
            else:
                f.write("No large temperature jumps detected.\n")

        # Step 7: Backup CSV
        backup_path = os.path.join(output_dir, os.path.basename(csv_path))
        shutil.copy(csv_path, backup_path)

        summary["status"] = "success"
        summary["report_path"] = report_path
        summary["log_summary"] = log_summary
        summary["temperature_jumps"] = len(jumps)

    except Exception as e:
        summary["status"] = f"failed: {e}"
        # Write failure report
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(f"Pipeline failed: {e}\n")

    finally:
        print(f"Pipeline complete. Report: {report_path}")

    return summary


result = run_monitoring_pipeline(CSV_PATH, LOG_PATH, OUTPUT_DIR)
print("\nPipeline summary:")
for k, v in result.items():
    print(f"  {k:<22}: {v}")